In [6]:
import os
import google.generativeai as genai
from openai import OpenAI


from dotenv import load_dotenv

load_dotenv()

True

In [2]:
# gemini_api_key = os.getenv("GEMINI_API_KEY")
# # 1️⃣ Set your API key
# genai.configure(api_key= gemini_api_key)

# # 2️⃣ Choose the Gemini model (e.g., gemini-2.5-flash or gemini-2.5-pro)
# model = genai.GenerativeModel("gemini-2.5-flash")

In [7]:
# 1️⃣ Get API key (same pattern as GEMINI)
openai_api_key = os.getenv("OPENAI_API_KEY")

# 2️⃣ Initialize client
client = OpenAI(api_key=openai_api_key)

In [9]:
# 3️⃣ Prompt (same as yours)
prompt = """
What is the deal of Anthropic with Microsoft and Nvidia?

What is the deal?

Using queries and links, summarize and give the output.
"""

# 4️⃣ Generate response
response = client.responses.create(
    model="gpt-5-mini",
    input=prompt
)

# 5️⃣ Print response
print("\n🧠 OpenAI GPT-5 Mini Response:\n")
print(response.output_text)


🧠 OpenAI GPT-5 Mini Response:

I don’t have live web access in this chat to run searches and return current links. I can, however:

- Summarize the deals between Anthropic, Microsoft, and NVIDIA based on public reporting up through my last update (June 2024), and
- Give a short list of precise search queries and recommended reputable sources you can open to read the original coverage (so you can quickly pull up the articles and links).

If you want, I can then fetch and cite current links if you enable web browsing or paste a few articles here. Below are the summary and the suggested queries/sources.

Short summary (high level)
- Microsoft <> Anthropic
  - Microsoft entered a multiyear, multibillion-dollar partnership with Anthropic to invest in Anthropic and make Microsoft Azure Anthropic’s preferred cloud. The partnership is about providing cloud compute, distribution, and product integration: Anthropic’s models (Claude) are made available to enterprise customers through Azure and i

In [4]:
# Bag of Words
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ---------------------------
# Example Documents
# ---------------------------
documents = [
    "AI is transforming the world",
    "Machine learning is a part of AI",
    "Cooking pasta is easy and fun",
    "AI will shape the future",
    "Pasta recipes are very popular"
]

# ---------------------------
# User Input Sentence
# ---------------------------
user_sentence = "future of AI"

# ---------------------------
# Step 1: Create Bag of Words
# ---------------------------
vectorizer = CountVectorizer()
bow_matrix = vectorizer.fit_transform(documents + [user_sentence])

# Last row is the user input vector
user_vec = bow_matrix[-1]
doc_vecs = bow_matrix[:-1]

# ---------------------------
# Step 2: Compute Cosine Similarity
# ---------------------------
similarities = cosine_similarity(user_vec, doc_vecs).flatten()

# ---------------------------
# Step 3: Get Closest Document
# ---------------------------
closest_index = similarities.argmax()
closest_score = similarities[closest_index]
closest_doc = documents[closest_index]

print("\nUser Query:", user_sentence)
print("Closest Match:", closest_doc)
print("Similarity Score:", round(closest_score, 4))



User Query: future of AI
Closest Match: AI will shape the future
Similarity Score: 0.5164


In [5]:
from sklearn.feature_extraction.text import CountVectorizer

# ---------------------------
# Documents + User Query
# ---------------------------
documents = [
    "AI is transforming the world",
    "Machine learning is a part of AI",
    "Cooking pasta is easy and fun",
    "AI will shape the future",
    "Pasta recipes are very popular"
]

user_query = "future of AI"

# Combine documents + query
all_texts = documents + [user_query]

# ---------------------------
# Create BoW Vectors
# ---------------------------
vectorizer = CountVectorizer()
bow_matrix = vectorizer.fit_transform(all_texts)

bow_array = bow_matrix.toarray()
vocab = vectorizer.get_feature_names_out()

# ---------------------------
# Print Vocabulary
# ---------------------------
print("Vocabulary:\n", vocab)

# ---------------------------
# Print document-wise BoW
# ---------------------------
print("\n--- Document-wise BoW Representation ---\n")
for idx, doc in enumerate(documents):
    print(f"Document {idx}: {doc}")
    print("BoW Vector:", bow_array[idx])
    print()

# ---------------------------
# Print User Query BoW
# ---------------------------
print("User Query:", user_query)
print("User Query BoW Vector:", bow_array[-1])
print("\nWord → Count in User Query:")
for word, count in zip(vocab, bow_array[-1]):
    if count > 0:
        print(f"{word}: {count}")


Vocabulary:
 ['ai' 'and' 'are' 'cooking' 'easy' 'fun' 'future' 'is' 'learning'
 'machine' 'of' 'part' 'pasta' 'popular' 'recipes' 'shape' 'the'
 'transforming' 'very' 'will' 'world']

--- Document-wise BoW Representation ---

Document 0: AI is transforming the world
BoW Vector: [1 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 1 1 0 0 1]

Document 1: Machine learning is a part of AI
BoW Vector: [1 0 0 0 0 0 0 1 1 1 1 1 0 0 0 0 0 0 0 0 0]

Document 2: Cooking pasta is easy and fun
BoW Vector: [0 1 0 1 1 1 0 1 0 0 0 0 1 0 0 0 0 0 0 0 0]

Document 3: AI will shape the future
BoW Vector: [1 0 0 0 0 0 1 0 0 0 0 0 0 0 0 1 1 0 0 1 0]

Document 4: Pasta recipes are very popular
BoW Vector: [0 0 1 0 0 0 0 0 0 0 0 0 1 1 1 0 0 0 1 0 0]

User Query: future of AI
User Query BoW Vector: [1 0 0 0 0 0 1 0 0 0 1 0 0 0 0 0 0 0 0 0 0]

Word → Count in User Query:
ai: 1
future: 1
of: 1


In [12]:
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

documents = [
    "Lions live in the wild and hunt for food",
    "Dogs are common indoor pets and are friendly",
    "Cars are used for transportation on roads",
    "Cats are also popular indoor pets",
    "Elephants live in the forest and are very large animals"
]

# tokenize
tokenized_docs = [simple_preprocess(doc) for doc in documents]

# train
model = Word2Vec(sentences=tokenized_docs, vector_size=50, min_count=1)

# average embedding
def get_avg_embedding(model, doc_tokens):
    vectors = [model.wv[w] for w in doc_tokens if w in model.wv]
    return np.mean(vectors, axis=0)

# all document embeddings
doc_embeddings = np.array([get_avg_embedding(model, doc) for doc in tokenized_docs])

# --- Input ---
query_raw = "pet"
query_tokens = simple_preprocess(query_raw)   # → ["pet"]

if query_tokens and query_tokens[0] in model.wv:
    query_embedding = model.wv[query_tokens[0]]
else:
    query_embedding = np.zeros(model.vector_size)

# similarity
sims = cosine_similarity([query_embedding], doc_embeddings)[0]
best_idx = np.argmax(sims)

print("Query word:", query_raw)
print("\nDocument Ranking:")
for score, doc in sorted(zip(sims, documents), reverse=True):
    print(f"{score:.3f} -> {doc}")

print("\nMOST relevant document:")
print(documents[best_idx])

Query word: pet

Document Ranking:
0.000 -> Lions live in the wild and hunt for food
0.000 -> Elephants live in the forest and are very large animals
0.000 -> Dogs are common indoor pets and are friendly
0.000 -> Cats are also popular indoor pets
0.000 -> Cars are used for transportation on roads

MOST relevant document:
Lions live in the wild and hunt for food


In [7]:
query_embedding

array([-0.01724647,  0.00734008,  0.01037878,  0.01148256,  0.0149413 ,
       -0.01234019,  0.00221531,  0.01209529, -0.00568585, -0.01234576,
       -0.00081565, -0.01674025, -0.01118867,  0.01419871,  0.00671182,
        0.01445208,  0.01360262,  0.01506363, -0.00758801, -0.00112764,
        0.00469629, -0.00903253,  0.0167851 , -0.01972398,  0.01352854,
        0.00582873, -0.00985849,  0.00879369, -0.0034807 ,  0.01341765,
        0.0199317 , -0.00872945, -0.00119287, -0.0113812 ,  0.00769979,
        0.00558018,  0.01378867,  0.01219834,  0.01907063,  0.01855223,
        0.01579281, -0.0139733 , -0.01831043, -0.00070704, -0.00619038,
        0.01579152,  0.01187064, -0.00309455,  0.00301743,  0.00358184],
      dtype=float32)

In [10]:
sims = cosine_similarity([query_embedding], doc_embeddings)[0]
sims

array([0.0262416 , 0.42474532, 0.08038595, 0.5182874 , 0.10140795],
      dtype=float32)

In [11]:
np.argmax(sims)

np.int64(3)

In [13]:
doc_embeddings

array([[-3.4469077e-03,  2.8247396e-03, -7.1731815e-03, -7.9747075e-03,
         1.1942667e-03, -1.8594954e-03, -9.5847831e-04,  2.2185550e-03,
         1.0004594e-03,  1.7284941e-03, -1.7778805e-03,  7.8204606e-04,
         4.5177545e-03, -2.3147692e-03,  4.1397084e-03, -8.8700163e-04,
         1.0380222e-02,  7.6548285e-03, -5.7109687e-03, -1.8486852e-03,
         3.4598103e-03,  1.8449115e-03,  2.4488210e-03,  3.4473743e-03,
         3.0606973e-03,  5.9026657e-03,  2.4056807e-03,  4.1870121e-03,
        -4.5594028e-03,  1.9686474e-03, -1.2275926e-03, -6.9730397e-04,
         7.7826116e-04,  3.3849676e-03, -2.5269184e-03, -2.0258473e-03,
         4.3000989e-03, -1.1598913e-03, -1.6777787e-03, -3.0523259e-04,
        -1.2809822e-03,  4.2998241e-03,  6.1712409e-03, -1.8584462e-03,
         5.4695988e-03, -8.7867916e-04,  6.6405330e-03, -4.9556186e-03,
        -1.5458009e-04,  2.3304755e-03],
       [-7.2960701e-04,  1.7511679e-04,  2.8916639e-03,  9.9057779e-03,
        -1.4834732e-03,

In [41]:
query_tokens

['large']

In [42]:
query_embedding

array([ 0.00804228,  0.00871828,  0.01990681, -0.00895048, -0.00275748,
       -0.0146388 , -0.0193977 , -0.01814892, -0.00206197, -0.01300747,
        0.00970824, -0.01233231,  0.0050698 ,  0.00146554, -0.00676142,
       -0.00196962,  0.01998079,  0.01830931, -0.0089573 ,  0.01815568,
       -0.01129314,  0.01185967, -0.00618441,  0.00684673,  0.00604211,
        0.01379513, -0.00472211,  0.01756527,  0.01516998, -0.01909332,
       -0.01600644, -0.01529948,  0.00583796, -0.00558885, -0.01387321,
       -0.01624967,  0.01663579,  0.00397704, -0.01866848, -0.00956976,
        0.00626166, -0.00941163,  0.01056324, -0.00845195,  0.00531032,
       -0.01608006,  0.01241373,  0.0096293 ,  0.00156384,  0.00602951],
      dtype=float32)

In [13]:
# Pre-trained model
import gensim.downloader as api

model = api.load("word2vec-google-news-300")

In [21]:
print(model.most_similar("cars"))

[('vehicles', 0.8008111715316772), ('car', 0.7423830032348633), ('automobiles', 0.7095544338226318), ('Cars', 0.6786174178123474), ('motorcycles', 0.6766677498817444), ('trucks', 0.6515100002288818), ('Porsches', 0.6339792013168335), ('bikes', 0.6299718022346497), ('BMWs', 0.6202709674835205), ('SUVs', 0.6192982196807861)]


In [17]:
print(model.similarity("cars", "automobiles"))

0.70955455


In [5]:
#print(model["carsvehicles"].shape)

In [24]:
from gensim.models import FastText
from gensim.utils import simple_preprocess
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

documents = [
    "Lion live in the wild and hunt for food",
    "Dogs are common indoor pets and are friendly",
    "Cars are used for transportation on roads",
    "Cats are also popular indoor pets",
    "Elephants live in the forest and are very large animals"
]

# tokenize
tokenized_docs = [simple_preprocess(doc) for doc in documents]

# Train FastText
model = FastText(
    sentences=tokenized_docs,
    vector_size=50
)

# average embedding
def get_avg_embedding(model, doc_tokens):
    vectors = [model.wv[w] for w in doc_tokens]
    return np.mean(vectors, axis=0)

# all document embeddings
doc_embeddings = np.array([get_avg_embedding(model, doc) for doc in tokenized_docs])

# query
query_raw = "pet"
query_tokens = simple_preprocess(query_raw)   # -> ['pet']

# FastText can ALWAYS generate embedding for unseen words (subword)
query_embedding = model.wv[query_tokens[0]]

# similarity
sims = cosine_similarity([query_embedding], doc_embeddings)[0]
best_idx = np.argmax(sims)

print("Query word:", query_raw)
print("\nDocument Ranking:")
for score, doc in sorted(zip(sims, documents), reverse=True):
    print(f"{score:.3f} -> {doc}")

print("\nMOST relevant document:")
print(documents[best_idx])

Query word: pet

Document Ranking:
0.298 -> Dogs are common indoor pets and are friendly
0.280 -> Cats are also popular indoor pets
0.126 -> Cars are used for transportation on roads
-0.115 -> Elephants live in the forest and are very large animals
-0.169 -> Lions live in the wild and hunt for food

MOST relevant document:
Dogs are common indoor pets and are friendly


In [49]:
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess

# Training sentences
sentences = [
    simple_preprocess("I am sitting on the river bank"),
    simple_preprocess("I visit the bank to deposit money")
]

# Train Word2Vec (tiny toy model)
model = Word2Vec(sentences=sentences, vector_size=20, window=3, min_count=1)

# SAME embedding for "bank"
bank1 = model.wv["bank"]    # from river sentence
bank2 = model.wv["bank"]    # from money sentence

print("Word2Vec - bank embedding (context-free):")
print(bank1)

# Verify BOTH embeddings are exactly same
import numpy as np

print("\nAre both embeddings identical?")
print(np.allclose(bank1, bank2))


Word2Vec - bank embedding (context-free):
[-0.00268114  0.00118216  0.02551675  0.04504637 -0.04651475 -0.03558404
  0.03229436  0.04486494 -0.02507714 -0.01881686  0.03690252 -0.00766736
 -0.02268307  0.03277026 -0.0243008  -0.00908009  0.0143829   0.00495937
 -0.04142607 -0.04724409]

Are both embeddings identical?
True


In [25]:
from transformers import BertTokenizer, BertModel
import torch
import numpy as np

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertModel.from_pretrained("bert-base-uncased")

sentence1 = "I am sitting on the river bank"
sentence2 = "I visit the bank to deposit money"

# Encode
inputs1 = tokenizer(sentence1, return_tensors="pt")
inputs2 = tokenizer(sentence2, return_tensors="pt")

# Get token embeddings
with torch.no_grad():
    outputs1 = model(**inputs1)
    outputs2 = model(**inputs2)

# Pick the token embedding for the word "bank"
tokens1 = tokenizer.tokenize(sentence1)
tokens2 = tokenizer.tokenize(sentence2)

# find index of 'bank'
idx1 = tokens1.index("bank")
idx2 = tokens2.index("bank")

bank_emb1 = outputs1.last_hidden_state[0][idx1].numpy()
bank_emb2 = outputs2.last_hidden_state[0][idx2].numpy()

print("BERT embedding for 'bank' in river sentence:")
print(bank_emb1)

print("\nBERT embedding for 'bank' in money sentence:")
print(bank_emb2)

# similarity check
from sklearn.metrics.pairwise import cosine_similarity

sim = cosine_similarity([bank_emb1], [bank_emb2])[0][0]
print("\nCosine similarity between two 'bank' embeddings:", sim)


BERT embedding for 'bank' in river sentence:
[ 3.28131139e-01  2.09828198e-01 -1.81585968e-01 -3.64794344e-01
  1.68668151e-01 -2.08227023e-01  5.86012661e-01  9.97573018e-01
 -6.01028383e-01 -4.10039485e-01  1.78537533e-01 -3.74858171e-01
 -2.28934705e-01  4.47715670e-01 -6.69759154e-01  6.03850126e-01
  4.79483865e-02  5.16201183e-02  9.20938969e-01  7.24657059e-01
 -7.79243529e-01  4.69402522e-01 -1.04568012e-01  6.95001066e-01
  8.65877092e-01  1.73293307e-01  3.85516524e-01 -4.22763914e-01
 -1.96797162e-01 -4.79022473e-01  8.72377992e-01  1.81074739e-01
  3.06429476e-01 -2.30069339e-01 -3.61418158e-01 -4.36996579e-01
 -5.35221457e-01 -4.04529750e-01 -6.89916611e-01  3.56537625e-02
 -3.17005754e-01 -5.77179730e-01 -5.42803347e-01  4.07654017e-01
  7.09930539e-01  9.33698192e-03  8.03048790e-01  2.89463431e-01
  5.42830884e-01  2.28726089e-01 -5.75697005e-01  1.02791095e+00
 -6.49315476e-01 -8.84129167e-01 -4.86422420e-01  7.35208869e-01
 -2.40012929e-02 -6.19253218e-01 -1.37023151e

In [26]:
bank_emb1.shape

(768,)

In [10]:
!pip install sentence-transformers numpy

In [22]:
#Sentence embeddings
# pip install sentence-transformers numpy scikit-learn

from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# 1) Tiny KB (10 sentences)
docs = [
    "AI automates repetitive tasks and saves time.",
    "AI works continuously without breaks.",
    "AI quickly detects patterns in large datasets.",
    "AI improves decision-making with data insights.",
    "AI personalizes user experiences across apps.",
    "AI reduces human error in routine processes.",
    "AI scales services to millions of users.",
    "AI speeds up support with smart chatbots.",
    "AI lowers operational costs via efficiency.",
    "AI enables rapid prototyping and experimentation.", 
    "Electric Cars are the future", 
    "India is trying to adopt electric",
]

# 2) Load a lightweight BERT embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# 3) Precompute document embeddings (no normalization here)
doc_embs = model.encode(docs)

# 4) Query → embed → cosine sims → top-2
query = "What are the benefits AI?"
q = model.encode([query])[0]

# Compute cosine similarity
sims = cosine_similarity([q], doc_embs)[0]

top2 = sims.argsort()[-2:][::-1]

# 5) Retrieved context (top-2)
context = [docs[i] for i in top2]

print("Query:", query)
print("\nTop-2 sentences:")
for i in top2:
    print(f"- ({sims[i]:.3f}) {docs[i]}")


Query: What are the benefits AI?

Top-2 sentences:
- (0.670) AI lowers operational costs via efficiency.
- (0.628) AI reduces human error in routine processes.


In [23]:
sims

array([0.6047861 , 0.5000106 , 0.3639807 , 0.59473366, 0.46440026,
       0.6280563 , 0.5898248 , 0.4706139 , 0.67011404, 0.5758702 ,
       0.23178835, 0.17690301], dtype=float32)

In [54]:
# 4) Query → embed → cosine sims → top-2
query = "What are the benefits of AI?"

Query: What are the benefits of AI?
Top-2 sentences:
- (0.642) AI lowers operational costs via efficiency.
- (0.623) AI reduces human error in routine processes.

Answer (from retrieved):
AI lowers operational costs via efficiency. AI reduces human error in routine processes.


In [18]:
# rag with explicit cosine similarity

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 1) Tiny knowledge base
documents = [
    "AI automates repetitive tasks and saves time.",
    "AI works continuously without breaks.",
    "AI quickly detects patterns in large datasets.",
    "AI improves decision-making with data insights.",
    "AI personalizes user experiences across apps.",
    "AI reduces human error in routine processes.",
    "AI scales services to millions of users.",
    "AI speeds up support with smart chatbots.",
    "AI lowers operational costs via efficiency.",
    "AI enables rapid prototyping and experimentation."
]

# 2) Load embedding model (Sentence-BERT)
model = SentenceTransformer("all-MiniLM-L6-v2")

# 3) Encode documents (no normalization)
doc_embs = model.encode(documents)

# 4) Query → embedding
query = "What are the benefits of AI?"
q_vec = model.encode([query])[0]

# 5) Compute cosine similarity explicitly
sims = cosine_similarity([q_vec], doc_embs)[0]

# Top-2 most similar docs
top2_idx = sims.argsort()[-2:][::-1]
retrieved = [documents[i] for i in top2_idx]

# 6) Build RAG prompt
prompt = f"""Answer the question using ONLY these sentences:

- {retrieved[0]}
- {retrieved[1]}

Question: {query}
Answer briefly:"""

print("Query:", query)
print("Top-2 sentences:")
for i in top2_idx:
    print(f"- ({sims[i]:.3f}) {documents[i]}")

print("\nPrompt to LLM:\n", prompt)


Query: What are the benefits of AI?
Top-2 sentences:
- (0.642) AI lowers operational costs via efficiency.
- (0.623) AI reduces human error in routine processes.

Prompt to LLM:
 Answer the question using ONLY these sentences:

- AI lowers operational costs via efficiency.
- AI reduces human error in routine processes.

Question: What are the benefits of AI?
Answer briefly:


In [57]:
# 1️⃣ Set your API key
genai.configure(api_key= gemini_api_key)

# 2️⃣ Choose the Gemini model (e.g., gemini-2.5-flash or gemini-2.5-pro)
model = genai.GenerativeModel("gemini-2.5-flash")
# 4️⃣ Generate the response
response = model.generate_content(prompt)

# https://generativelanguage.googleapis.com/v1beta/{model="gemini-2.5-flash"}:generateContent::"":

# 5️⃣ Print the model’s reply
print("\n🧠 Gemini 2.5 Response:\n")
print(response.text)


🧠 Gemini 2.5 Response:

AI lowers operational costs via efficiency and reduces human error.


In [45]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 1) Tiny knowledge base
documents = [
    "AI automates repetitive tasks and saves time.",
    "AI works continuously without breaks.",
    "AI quickly detects patterns in large datasets.",
    "AI improves decision-making with data insights.",
    "AI personalizes user experiences across apps.",
    "AI reduces human error in routine processes.",
    "AI scales services to millions of users.",
    "AI speeds up support with smart chatbots.",
    "AI lowers operational costs via efficiency.",
    "AI enables rapid prototyping and experimentation."
]

# 2) Load embedding model (Sentence-BERT)
model = SentenceTransformer("all-MiniLM-L6-v2")

# 3) Encode documents (no normalization)
doc_embs = model.encode(documents)

# 4) Query → embedding
#query = "What are the benefits of AI?"
query = input("Enter User query")
q_vec = model.encode([query])[0]

# 5) Compute cosine similarity explicitly
sims = cosine_similarity([q_vec], doc_embs)[0]

# Top-2 most similar docs
top2_idx = sims.argsort()[-2:][::-1]
retrieved = [documents[i] for i in top2_idx]
retrieved

Enter User query what are benefits oF AI


['AI lowers operational costs via efficiency.',
 'AI reduces human error in routine processes.']

In [44]:
retrieved

['AI lowers operational costs via efficiency.',
 'AI reduces human error in routine processes.']

In [46]:
prompt = f"""
Answer the following question by using ONLY the below contenxt 

sen1 : {retrieved[0]}
sen2 : {retrieved[1]}

question = {query}
Answer briefly
"""

In [47]:
prompt

'\nAnswer the following question by using ONLY the below contenxt \n\nsen1 : AI lowers operational costs via efficiency.\nsen2 : AI reduces human error in routine processes.\n\nquestion = what are benefits oF AI\nAnswer briefly\n'

In [28]:
# 3️⃣ Prompt (same as yours)
prompt = """
Answer the following question by using ONLY the below contenxt 

sen1 : AI lowers operational costs via efficiency.
sen2 : AI reduces human error in routine processes

question = What are the benefits of AI?
Answer briefly
"""

# 4️⃣ Generate response
response = client.responses.create(
    model="gpt-5-mini",
    input=prompt
)

# 5️⃣ Print response
print("\n🧠 OpenAI GPT-5 Mini Response:\n")
print(response.output_text)


🧠 OpenAI GPT-5 Mini Response:

- Lowers operational costs via increased efficiency.
- Reduces human error in routine processes.


In [27]:
# 3️⃣ Prompt (same as yours)
prompt = """
What are the benefits of AI?
"""

# 4️⃣ Generate response
response = client.responses.create(
    model="gpt-5-mini",
    input=prompt
)

# 5️⃣ Print response
print("\n🧠 OpenAI GPT-5 Mini Response:\n")
print(response.output_text)


🧠 OpenAI GPT-5 Mini Response:

AI brings many benefits across individuals, organizations, and society. Key advantages include:

- Improved efficiency and automation  
  AI automates repetitive, time-consuming tasks (e.g., data entry, scheduling, manufacturing), freeing people for higher-value work and reducing error rates.

- Faster, better decisions from data  
  Machine learning can analyze large, complex datasets to surface patterns and predictions (e.g., sales forecasting, demand planning, risk assessment) that humans alone would miss.

- Enhanced healthcare  
  AI aids diagnostics (image analysis, symptom triage), personalizes treatment plans, speeds drug discovery, and helps monitor patients remotely, improving outcomes and access to care.

- Personalized experiences and services  
  Recommendation systems and adaptive interfaces tailor content, learning, and services to individual needs (e.g., personalized education, shopping, news feeds).

- Increased safety and reliability  


In [ ]:
PINECONE_API = pcsk_7Pvst8_AA1AdSpB14RPj4ZL2eV1X2QE1Q768XHfanzdBDprUURmjBNv7Qsi5rYNEkxTaHr